# 04 - Full Dataset Mass Inference

Classifies the **entire remaining tweet corpus** (~17 million tweets) using the
models trained in notebook 02. Models are loaded from `Models/Classifiers/`.

This notebook is designed to run on Colab with GPU acceleration and processes tweets
in chunks to avoid out-of-memory errors.

In [ ]:
import os
from pathlib import Path
import sys

# --- ENVIRONMENT SWITCH ---
# True  → local machine with Google Drive Desktop mounted
# False → Google Colab cloud
RUNNING_LOCALLY = False

if RUNNING_LOCALLY:
    _REPO_ROOT = str(Path(os.getcwd()).resolve().parents[1])
    if _REPO_ROOT not in sys.path:
        sys.path.insert(0, _REPO_ROOT)
    BASE_PATH = Path('/Volumes/GoogleDrive/My Drive/Colab Projects/AI Public Trust')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/My Drive/Colab Projects/AI Public Trust')

# Pre-compute critical paths
twits_folder          = BASE_PATH / 'Raw Data/Twits/'
test_folder           = BASE_PATH / 'Raw Data/'
datasets_folder       = BASE_PATH / 'Data Sets'
cleanedds_folder      = BASE_PATH / 'Data Sets/Cleaned Data'
networks_folder       = BASE_PATH / 'Data Sets/Networks/'
literature_folder     = BASE_PATH / 'Literature/'
topic_models_folder   = BASE_PATH / 'Models/Topic Modeling/'
classifiers_folder    = BASE_PATH / 'Models/Classifiers/'
classifiers_folder.mkdir(parents=True, exist_ok=True)
hitl_folder        = datasets_folder / 'Classifiers_Data' / 'HITL'
full_inference_folder = datasets_folder / 'Classifiers_Data' / 'Full_Inference'
full_inference_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
if not RUNNING_LOCALLY:
    print('Running Colab setup...')
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'transformers', 'torch',
                    'sentence-transformers', 'lightgbm'])
else:
    print('Running locally: skipping Colab setup.')

In [ ]:
import os
import re
import time
import pickle
import numpy as np
import pandas as pd
import tqdm
import torch
import lightgbm as lgb
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

## Configuration

Set `CHUNK_SIZE` based on available RAM. 10 000 is safe for Colab; increase if you have more memory.

In [ ]:
CHUNK_SIZE    = 10_000   # tweets processed per batch
ROBERTA_BATCH = 64       # pipeline batch size for RoBERTa (GPU dependent)
SAVE_EVERY    = 100_000  # checkpoint: save intermediate results every N tweets

## 1. Text Cleaning

In [ ]:
def clean_tweet(text: str) -> str:
    text = re.sub(r'http\S+', '', text)          # remove URLs
    text = re.sub(r'@[A-Za-z0-9_]+', '', text)   # remove @mentions
    text = text.replace('#', '')                  # strip # from hashtags
    text = re.sub(r'\bRT\b', '', text)           # remove RT markers
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

## 2. Load Full Tweet Dataset

Expects the full cleaned tweet corpus as a DataFrame or pickle dict.
Adjust the path and loading logic to match your data format.

In [ ]:
# ── Adjust this path and format to your actual full corpus ──────────────
FULL_DATA_PATH = cleanedds_folder / 'all_cleaned_tweets.pkl'

print(f'Loading full corpus from {FULL_DATA_PATH}...')
if FULL_DATA_PATH.suffix == '.pkl':
    full_df = pd.read_pickle(FULL_DATA_PATH)
elif FULL_DATA_PATH.suffix == '.csv':
    full_df = pd.read_csv(FULL_DATA_PATH)
else:
    raise ValueError('Unknown format. Update FULL_DATA_PATH.')

# Ensure id column exists
if 'tweet_id' in full_df.columns and 'id' not in full_df.columns:
    full_df['id'] = full_df['tweet_id']

print(f'Total tweets to classify: {len(full_df):,}')

## 3. Load Trained Models

In [ ]:
# ── Twitter-RoBERTa (primary model) ─────────────────────────────────────
best_model_path = classifiers_folder / 'best_roberta_model'
if not best_model_path.exists():
    raise FileNotFoundError(f'RoBERTa model not found at {best_model_path}. Run notebook 02 first.')

print('Loading Twitter-RoBERTa...')
tokenizer = AutoTokenizer.from_pretrained(str(best_model_path))
model     = AutoModelForSequenceClassification.from_pretrained(str(best_model_path))
device    = 0 if torch.cuda.is_available() else -1
roberta_pipe = pipeline('text-classification', model=model, tokenizer=tokenizer,
                        device=device, batch_size=ROBERTA_BATCH, return_all_scores=True)
print(f'RoBERTa loaded. Device: {"GPU" if device==0 else "CPU"}')

In [ ]:
# ── LightGBM (sentence embeddings) ──────────────────────────────────────
lgb_embed_path = classifiers_folder / 'lgb_embed.txt'
lgb_bow_path   = classifiers_folder / 'lgb_bow.txt'

clf_embed = lgb.Booster(model_file=str(lgb_embed_path)) if lgb_embed_path.exists() else None
clf_bow   = lgb.Booster(model_file=str(lgb_bow_path))   if lgb_bow_path.exists()   else None

if clf_embed:
    print('LightGBM (embed) loaded.')
if clf_bow:
    print('LightGBM (BoW) loaded.')

In [ ]:
# ── Sentence Transformer (for LightGBM embed) ───────────────────────────
if clf_embed:
    st_model = SentenceTransformer('all-MiniLM-L6-v2')
    print('Sentence transformer loaded.')

## 4. Run Mass Inference

Processes tweets in chunks. Results are saved incrementally every `SAVE_EVERY` tweets
to avoid losing progress on long runs.

In [ ]:
results = []
t_start = time.time()
n_total = len(full_df)

for chunk_start in tqdm.tqdm(range(0, n_total, CHUNK_SIZE)):
    chunk = full_df.iloc[chunk_start:chunk_start + CHUNK_SIZE].copy()
    texts_raw = chunk['text'].astype(str).tolist()
    texts_clean = [clean_tweet(t) for t in texts_raw]

    row_results = {'id': chunk['id'].tolist()
                   if 'id' in chunk.columns else list(range(chunk_start, chunk_start+len(chunk)))}

    # ── RoBERTa predictions ──────────────────────────────────────────
    scores = roberta_pipe(texts_clean)
    row_results['roberta_label']      = [max(s, key=lambda x: x['score'])['label'] for s in scores]
    row_results['roberta_confidence'] = [max(s, key=lambda x: x['score'])['score'] for s in scores]

    # ── LightGBM (embed) predictions ────────────────────────────────
    if clf_embed:
        embeds = st_model.encode(texts_clean, show_progress_bar=False)
        lgb_embed_probs = clf_embed.predict(embeds)
        row_results['lgb_embed_label'] = (lgb_embed_probs > 0.5).astype(int).tolist()
        row_results['lgb_embed_prob']  = lgb_embed_probs.tolist()

    results.append(pd.DataFrame(row_results))

    # ── Checkpoint save ──────────────────────────────────────────────
    processed_so_far = chunk_start + len(chunk)
    if processed_so_far % SAVE_EVERY < CHUNK_SIZE:
        checkpoint_df = pd.concat(results, ignore_index=True)
        checkpoint_path = full_inference_folder / f'checkpoint_{processed_so_far}.pkl'
        checkpoint_df.to_pickle(checkpoint_path)
        print(f'Checkpoint saved at {processed_so_far:,} tweets ({time.time()-t_start:.0f}s elapsed)')

print(f'
Done. Total time: {time.time()-t_start:.1f}s')
results_df = pd.concat(results, ignore_index=True)

## 5. Merge Predictions Back and Save Final Output

In [ ]:
# Merge predictions onto the original dataframe (keeping all original columns)
final_df = full_df.merge(results_df, on='id', how='left')

print(f'Final annotated dataset: {len(final_df):,} rows')
print(final_df[['id', 'text', 'roberta_label', 'roberta_confidence']].head())

In [ ]:
# Save as both pickle (fast I/O) and CSV (interoperability)
out_pkl = full_inference_folder / 'full_inference_annotated.pkl'
out_csv = full_inference_folder / 'full_inference_annotated.csv'

final_df.to_pickle(out_pkl)
final_df.to_csv(out_csv, index=False)
print(f'Saved to:
  {out_pkl}
  {out_csv}')